1. Loading candidate profiles from a parquet file
2. Converting each row into a clean profile string
3. Embedding profile text
4. Preparing records for vector databases (Qdrant / Elasticsearch)


In [14]:
import pandas as pd
import pyarrow.parquet as pq
from typing import Iterable, List, Optional, TypeGuard

DATASET_PATH = '../data/candidates.parquet'
ID_COLUMN = 'id'                            # update if needed
MAX_ROWS = 5                               # set e.g. 1000 while iterating or None for all


In [15]:
# Use pyarrow directly: pd.read_parquet re-imports pandas extension types and can
# fail in a long-lived notebook kernel with "pandas.period already defined".
df = pq.read_table(DATASET_PATH).to_pandas()
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

print(f'Loaded rows: {len(df):,}')
display(df.head(3))
print('Columns:', list(df.columns))


Loaded rows: 5


,Position,Moreinfo,Looking For,Highlights,Primary Keyword,English Level,Experience Years,CV,CV_lang,id,__index_level_0__
0,"13 years of exp || Solidity, C#, JavaScript ||...",Who am I:\r\n- 13 years of commercial experien...,I am interested in:\r\n- part-time engagement;...,Landed a role of Director of Blockchain Develo...,Lead,fluent,11.0,Landed a role of Director of Blockchain Develo...,en,50534b61-6826-52b1-9ac5-bfd2cfa348ec,24230
1,1c Developer,Worked on a mobile application for tracking trips,NaN,NaN,Other,intermediate,3.0,\nWorked on a mobile application for tracking ...,en,c2b9ea56-b5c8-50ec-a63b-053a5c8ff467,24231
2,1C developer,1 am an 1C developer. I deployed an 1C to typo...,NaN,NaN,Flutter,intermediate,11.0,\n1 am an 1C developer. I deployed an 1C to ty...,en,b3bfe3ed-ec25-56b2-8aaa-8c629120538e,24232


Columns: ['Position', 'Moreinfo', 'Looking For', 'Highlights', 'Primary Keyword', 'English Level', 'Experience Years', 'CV', 'CV_lang', 'id', '__index_level_0__']


In [16]:
def is_missing(value: object) -> bool:
    if value is None:
        return True
    if isinstance(value, float):
        return value != value
    missing = pd.isna(value)
    return bool(missing) if isinstance(missing, bool) else False

def is_present(value: object) -> TypeGuard[float | int | str]:
    return not is_missing(value)

def normalize_text(value) -> str:
    if is_missing(value):
        return ''
    text = str(value).strip()
    return ' '.join(text.split())

def join_if_present(label: str, values: Iterable[str]) -> str:
    cleaned = [v for v in values if v]
    if not cleaned:
        return ''
    return f"{label}: " + '; '.join(cleaned)

def bio_fields(moreinfo: str, cv: str) -> tuple[str, Optional[str]]:
    """Pick one Bio section; return extra CV section only when it adds new content."""
    if not cv or cv == moreinfo:
        return moreinfo, None
    if not moreinfo:
        return cv, None
    bio = cv if len(cv) > len(moreinfo) else moreinfo
    extra_cv = cv if cv != bio else None
    return bio, extra_cv

def row_to_profile(row: pd.Series) -> str:
    position = normalize_text(row.get('Position', ''))
    moreinfo = normalize_text(row.get('Moreinfo', ''))
    looking_for = normalize_text(row.get('Looking For', ''))
    highlights = normalize_text(row.get('Highlights', ''))
    english_level = normalize_text(row.get('English Level', ''))

    experience_years_raw = row.get('Experience Years')
    if is_present(experience_years_raw):
        years = float(experience_years_raw)
        experience_years = f"{years:g} years" if years != 1 else "1 year"
    else:
        experience_years = ''

    cv = normalize_text(row.get('CV', ''))
    bio, extra_cv = bio_fields(moreinfo, cv)

    sections = [
        join_if_present('Position', [position]),
        join_if_present('Bio', [bio]),
        join_if_present('Looking for', [looking_for]),
        join_if_present('Highlights', [highlights]),
        join_if_present('English level', [english_level]),
        join_if_present('Experience', [experience_years]),
        join_if_present('CV', [extra_cv]) if extra_cv else '',
    ]
    return '\n'.join([s for s in sections if s])


In [18]:
df['profile_text'] = df.apply(row_to_profile, axis=1)
display(df[[ID_COLUMN, 'profile_text']].head(3))


,id,profile_text
0,50534b61-6826-52b1-9ac5-bfd2cfa348ec,"Position: 13 years of exp || Solidity, C#, Jav..."
1,c2b9ea56-b5c8-50ec-a63b-053a5c8ff467,Position: 1c Developer\nBio: Worked on a mobil...
2,b3bfe3ed-ec25-56b2-8aaa-8c629120538e,Position: 1C developer\nBio: 1 am an 1C develo...


In [ ]:
# Option A: local embedding model (pip install sentence-transformers)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("microsoft/harrier-oss-v1-0.6b", model_kwargs={"dtype": "auto"})
candidates = df['profile_text'].tolist()
candidate_embeddings = model.encode(candidates, prompt_name="sts_query")
# print the first embedding and its corresponding candidate
print(candidate_embeddings[0])
print(candidate_embeddings[0].shape)
print(candidates[0] + '\n\n')
print(candidate_embeddings[1])
print(candidate_embeddings[1].shape)
print(candidates[1] + '\n\n')


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7566.10it/s]


In [22]:
print(candidate_embeddings.shape)
print(candidate_embeddings[0].shape)

NameError: name 'candidates_embeddings' is not defined

In [ ]:
# Build payloads for vector DB upserts
# Assumes `vectors` aligns with df rows

records = []
# for idx, row in df.reset_index(drop=True).iterrows():
#     records.append({
#         'id': str(row[ID_COLUMN]),
#         'vector': vectors[idx],
#         'payload': {
#             'profile_text': row['profile_text'],
#             'title': row.get('title'),
#             'skills': row.get('skills'),
#         },
#     })

print('Starter payload template ready.')


In [ ]:
# Qdrant starter (pip install qdrant-client)
# from qdrant_client import QdrantClient
# from qdrant_client.http.models import Distance, VectorParams, PointStruct
#
# client = QdrantClient(url='http://localhost:6333')
# collection_name = 'candidate_profiles'
#
# client.recreate_collection(
#     collection_name=collection_name,
#     vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE),
# )
#
# points = [PointStruct(id=r['id'], vector=r['vector'], payload=r['payload']) for r in records]
# client.upsert(collection_name=collection_name, points=points)


In [ ]:
# Elasticsearch starter (pip install elasticsearch)
# from elasticsearch import Elasticsearch
#
# es = Elasticsearch('http://localhost:9200')
# index_name = 'candidate_profiles'
#
# for r in records:
#     doc = dict(r['payload'])
#     doc['vector'] = r['vector']
#     es.index(index=index_name, id=r['id'], document=doc)
#
# print('Indexed records in Elasticsearch')
